# H2: Vigor Dynamics Across the Threat Imminence Continuum

**Claim:** Threat modulates motor vigor independently of choice, with distinct dynamics across the predatory imminence continuum.

- **H2a:** Threat increases pressing rate within cookie type
- **H2b:** Predator encounter triggers a rapid motor spike
- **H2c:** Threat and encounter have distinct temporal signatures (GAM)

In [1]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, ast, os
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf
from patsy import dmatrix
from scipy.stats import pearsonr, ttest_rel, ttest_1samp, chi2, linregress
from config import *
from load_data import load_both

%matplotlib inline
plt.rcParams.update({'figure.dpi': 120, 'font.size': 10,
                     'axes.spines.right': False, 'axes.spines.top': False})

BIN = 0.2; SMOOTH = 3; K_SPLINE = 10
def smooth(s):
    return pd.Series(s).rolling(SMOOTH, center=True, min_periods=1).mean().values

exp_data, conf_data = load_both()
all_data = {'exploratory': exp_data, 'confirmatory': conf_data}
all_data = {k: v for k, v in all_data.items() if v is not None}

for name, d in all_data.items():
    s = d['config']
    n_metrics = len(d.get('vigor_metrics', pd.DataFrame()))
    print(f'{s.label}: {len(d["beh"])} trials, metrics={n_metrics}')


Repo root: /workspace
Loaded precomputed metrics: 102641 rows


Trials: 23490, Subjects: 290


## H2a: Threat increases pressing rate within cookie type

**Test:** Paired t-test, T=0.9 vs T=0.1, within heavy and within light cookies separately.
**Threshold:** p < .01 for both.

In [ ]:
# H2a: Threat increases pressing rate within cookie type
# Run for both samples

h2a_results = []
fig, axes = plt.subplots(2, 2, figsize=(12, 10), sharey='row')

for row_idx, (sample_name, d) in enumerate(all_data.items()):
    metrics = d.get('vigor_metrics')
    if metrics is None or len(metrics) == 0:
        print(f'{d["config"].label}: No vigor metrics — skipping H2a')
        continue
    
    full = metrics[metrics['epoch'] == 'full']
    label = d['config'].label
    
    for col_idx, (cookie, title) in enumerate([(1, 'Heavy (req=0.9)'), (0, 'Light (req=0.4)')]):
        ax = axes[row_idx, col_idx]
        sub = full[full['cookie'] == cookie]
        subj_means = sub.groupby(['subj', 'T_round'])['norm_rate'].mean().reset_index()
        cond = subj_means.groupby('T_round')['norm_rate'].agg(['mean', 'sem']).reset_index()
        
        colors = {0.1: '#2196F3', 0.5: '#9E9E9E', 0.9: '#F44336'}
        if len(cond) > 0:
            ax.bar(range(len(cond)), cond['mean'], yerr=cond['sem']*1.96,
                   color=[colors.get(t, '#999') for t in cond['T_round']],
                   edgecolor='black', linewidth=0.5, capsize=5, alpha=0.8)
            ax.set_xticks(range(len(cond)))
            ax.set_xticklabels([f'T={t}' for t in cond['T_round']])
        ax.set_title(f'{label}\n{title}', fontsize=10)
        if col_idx == 0:
            ax.set_ylabel('Normalized press rate')
    
    # Stats
    print(f"\nH2a — {label}:")
    print(f"{'Cookie':<8} {'M(T=.1)':>10} {'M(T=.9)':>10} {'Δ':>10} {'t':>8} {'p':>12} {'d':>8} {'Pass?':>6}")
    print("-" * 72)
    for cookie, clabel in [(1, 'Heavy'), (0, 'Light')]:
        sub = full[full['cookie'] == cookie]
        lo = sub[sub['T_round'] == 0.1].groupby('subj')['norm_rate'].mean()
        hi = sub[sub['T_round'] == 0.9].groupby('subj')['norm_rate'].mean()
        shared = sorted(set(lo.index) & set(hi.index))
        if len(shared) < 10: continue
        diff = hi.loc[shared].values - lo.loc[shared].values
        t, p = ttest_rel(hi.loc[shared], lo.loc[shared])
        d_val = diff.mean() / diff.std()
        passed = p < 0.01 and d_val > 0.15
        print(f"{clabel:<8} {lo.loc[shared].mean():>10.4f} {hi.loc[shared].mean():>10.4f} "
              f"{diff.mean():>+10.4f} {t:>8.2f} {p:>12.2e} {d_val:>+8.3f} {'✓' if passed else '✗':>6}")
        h2a_results.append({'sample': sample_name, 'cookie': clabel, 't': t, 'p': p, 'd': d_val, 'pass': passed})

plt.suptitle('H2a: Threat increases vigor within cookie type', fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(REPO_ROOT / 'results' / 'figs' / 'paper' / 'H2a_vigor_by_threat.png', dpi=150, bbox_inches='tight')
plt.show()

## H2b: Predator encounter triggers a rapid motor spike

**Test:** Encounter spike = attack minus non-attack press rate in reactive epoch. One-sample t vs zero.
**Threshold:** p < .001, d > 0.20.

In [ ]:
# H2b: Encounter spike — run for both samples
h2b_results = []

for sample_name, d in all_data.items():
    metrics = d.get('vigor_metrics')
    beh = d['beh']
    label = d['config'].label
    
    if metrics is None or len(metrics) == 0:
        print(f'{label}: No vigor metrics — skipping H2b')
        continue
    
    reactive = metrics[metrics['epoch'] == 'reactive']
    
    s_a = reactive[reactive['is_attack']==1].groupby('subj')['norm_rate'].mean()
    s_n = reactive[reactive['is_attack']==0].groupby('subj')['norm_rate'].mean()
    shared = sorted(set(s_a.index) & set(s_n.index))
    spike = s_a.loc[shared] - s_n.loc[shared]
    
    t_spike, p_spike = ttest_1samp(spike.dropna(), 0)
    d_spike = spike.mean() / spike.std()
    passed = p_spike < 0.001 and d_spike > 0.20
    
    print(f"\nH2b — {label}:")
    print(f"  Mean spike: {spike.mean():+.4f}")
    print(f"  t = {t_spike:.2f}, p = {p_spike:.2e}, d = {d_spike:+.3f}")
    print(f"  {(spike>0).mean()*100:.0f}% of subjects positive")
    print(f"  {'PASS ✓' if passed else 'FAIL ✗'}")
    
    # Threat independence
    spikes_by_T = {}
    for T in [0.1, 0.5, 0.9]:
        sa = reactive[(reactive['is_attack']==1)&(reactive['T_round']==T)].groupby('subj')['norm_rate'].mean()
        sn = reactive[(reactive['is_attack']==0)&(reactive['T_round']==T)].groupby('subj')['norm_rate'].mean()
        sh = sorted(set(sa.index) & set(sn.index))
        spikes_by_T[T] = sa.loc[sh] - sn.loc[sh]
    
    sh_all = sorted(set(spikes_by_T[0.1].index) & set(spikes_by_T[0.9].index))
    d01 = spikes_by_T[0.1].loc[sh_all]; d09 = spikes_by_T[0.9].loc[sh_all]
    clean = ~(d01.isna() | d09.isna())
    if clean.sum() > 10:
        t_mod, p_mod = ttest_rel(d09[clean], d01[clean])
        print(f"  Threat modulation: t={t_mod:.2f}, p={p_mod:.4f} (threat-independent: {'✓' if p_mod > 0.05 else '✗'})")
    
    h2b_results.append({'sample': sample_name, 't': t_spike, 'p': p_spike, 'd': d_spike, 'pass': passed})

## H2c: Threat and encounter have distinct temporal signatures

**Test:** GAM (spline basis + mixed model) with LRT for smooth-by-condition interactions.
**Threshold:** p < .01 for both encounter and threat LRTs.

In [ ]:
# H2c: GAM temporal signatures — run for both samples
# Uses epoch-level vigor metrics (anticipatory/reactive/terminal) rather than
# raw keypress timecourse, which requires the raw alignedEffortRate column.

h2c_results = []

for sample_name, d in all_data.items():
    metrics = d.get('vigor_metrics')
    label = d['config'].label
    
    if metrics is None or len(metrics) == 0:
        print(f'{label}: No vigor metrics — skipping H2c')
        continue
    
    # Use epoch-level data: each row is subj × trial × epoch
    # For GAM, we need time-binned data. Use epoch as a categorical time proxy.
    # Map epochs to approximate time: onset=0, anticipatory=1, reactive=2, terminal=3
    epoch_order = {'onset': 0, 'anticipatory': 1, 'reactive': 2, 'terminal': 3}
    gam_data = metrics[metrics['epoch'].isin(epoch_order.keys())].copy()
    gam_data['t_epoch'] = gam_data['epoch'].map(epoch_order)
    gam_data = gam_data[gam_data['norm_rate'].notna()]
    gam_data['cookie'] = gam_data['cookie'].astype(float)
    gam_data['is_attack'] = gam_data['is_attack'].astype(float)
    gam_data['T_round'] = gam_data['T_round'].round(1)
    
    # Aggregate to subj × epoch × condition
    agg = gam_data.groupby(['subj','t_epoch','is_attack','cookie']).agg(
        rate=('norm_rate','mean'), T_round=('T_round','mean')).reset_index()
    
    # Build spline basis on epoch time
    K = min(K_SPLINE, 4)  # only 4 epochs
    spline_mat = dmatrix(f'cr(t_epoch, df={K}) - 1', agg, return_type='dataframe')
    spl_names = [f'spl{i}' for i in range(K)]
    for i, col in enumerate(spline_mat.columns):
        agg[spl_names[i]] = spline_mat[col].values
    
    # Interaction: spline × is_attack
    atk_names = [f'a{i}' for i in range(K)]
    for i in range(K):
        agg[atk_names[i]] = agg[spl_names[i]] * agg['is_attack']
    
    spl_str = ' + '.join(spl_names)
    atk_str = ' + '.join(atk_names)
    
    # ENCOUNTER LRT
    f_base = f'rate ~ {spl_str} + cookie'
    f_full = f'rate ~ {spl_str} + {atk_str} + cookie'
    
    m_base = smf.mixedlm(f_base, agg, groups=agg['subj']).fit(reml=False)
    m_full = smf.mixedlm(f_full, agg, groups=agg['subj']).fit(reml=False)
    
    lr_enc = 2 * (m_full.llf - m_base.llf)
    p_enc = 1 - chi2.cdf(lr_enc, K)
    enc_pass = p_enc < 0.01
    
    print(f"\nH2c — {label}:")
    print(f"  Encounter LRT: χ²={lr_enc:.1f}, df={K}, p={p_enc:.2e} {'✓' if enc_pass else '✗'}")
    
    # THREAT LRT
    agg['T09'] = (agg['T_round'].round(1) >= 0.85).astype(float)
    agg['T05'] = ((agg['T_round'].round(1) >= 0.45) & (agg['T_round'].round(1) < 0.85)).astype(float)
    
    t09_names = [f't09_{i}' for i in range(K)]
    t05_names = [f't05_{i}' for i in range(K)]
    for i in range(K):
        agg[t09_names[i]] = agg[spl_names[i]] * agg['T09']
        agg[t05_names[i]] = agg[spl_names[i]] * agg['T05']
    
    t09_str = ' + '.join(t09_names)
    t05_str = ' + '.join(t05_names)
    
    f_base2 = f'rate ~ {spl_str} + cookie + is_attack'
    f_full2 = f'rate ~ {spl_str} + {t09_str} + {t05_str} + cookie + is_attack'
    
    m_base2 = smf.mixedlm(f_base2, agg, groups=agg['subj']).fit(reml=False)
    m_full2 = smf.mixedlm(f_full2, agg, groups=agg['subj']).fit(reml=False)
    
    lr_thr = 2 * (m_full2.llf - m_base2.llf)
    p_thr = 1 - chi2.cdf(lr_thr, 2 * K)
    thr_pass = p_thr < 0.01
    
    print(f"  Threat LRT:    χ²={lr_thr:.1f}, df={2*K}, p={p_thr:.2e} {'✓' if thr_pass else '✗'}")
    
    h2c_results.append({'sample': sample_name, 
                         'enc_chi2': lr_enc, 'enc_p': p_enc, 'enc_pass': enc_pass,
                         'thr_chi2': lr_thr, 'thr_p': p_thr, 'thr_pass': thr_pass})

In [ ]:
# H2 Summary — both samples
print('H2 SUMMARY')
print('=' * 80)
print(f'{"Test":<30} ', end='')
for name in all_data: print(f'{name:>20}', end='')
print()
print('-' * 80)

# H2a
for cookie in ['Heavy', 'Light']:
    print(f'H2a {cookie:<25} ', end='')
    for name in all_data:
        r = [x for x in h2a_results if x['sample']==name and x['cookie']==cookie]
        if r:
            print(f'{"✓" if r[0]["pass"] else "✗"} d={r[0]["d"]:+.2f}'.rjust(20), end='')
        else:
            print('N/A'.rjust(20), end='')
    print()

# H2b
print(f'H2b Encounter spike          ', end='')
for name in all_data:
    r = [x for x in h2b_results if x['sample']==name]
    if r:
        print(f'{"✓" if r[0]["pass"] else "✗"} d={r[0]["d"]:+.2f}'.rjust(20), end='')
    else:
        print('N/A'.rjust(20), end='')
print()

# H2c
for test, key in [('Encounter GAM', 'enc'), ('Threat GAM', 'thr')]:
    print(f'H2c {test:<25} ', end='')
    for name in all_data:
        r = [x for x in h2c_results if x['sample']==name]
        if r:
            print(f'{"✓" if r[0][f"{key}_pass"] else "✗"} χ²={r[0][f"{key}_chi2"]:.0f}'.rjust(20), end='')
        else:
            print('N/A'.rjust(20), end='')
    print()

# Overall
all_pass_exp = all(x['pass'] for x in h2a_results + h2b_results if x['sample']=='exploratory')
all_pass_conf = all(x['pass'] for x in h2a_results + h2b_results if x['sample']=='confirmatory')
enc_pass_exp = any(x['enc_pass'] for x in h2c_results if x['sample']=='exploratory')
thr_pass_exp = any(x['thr_pass'] for x in h2c_results if x['sample']=='exploratory')
enc_pass_conf = any(x['enc_pass'] for x in h2c_results if x['sample']=='confirmatory')
thr_pass_conf = any(x['thr_pass'] for x in h2c_results if x['sample']=='confirmatory')

print(f'\nOverall: Exploratory all pass: {all_pass_exp and enc_pass_exp and thr_pass_exp}')
print(f'         Confirmatory all pass: {all_pass_conf and enc_pass_conf and thr_pass_conf}')